# 27. Views

*Module: 7 — Advanced server · Companion summary: `lectures/27-views.md` · SQL engine: SQLite (in-memory)*

## What you will study

- create a virtual table with CREATE VIEW
- query a view like any table
- replace and drop views; honor the no-ORDER-BY-inside rule

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample table — EMPLOYEE with sensitive data

In [ ]:
q("CREATE TABLE employee (emp_id TEXT PRIMARY KEY, f_name TEXT, l_name TEXT, salary REAL, dep_id TEXT, ssn TEXT)")
q("""
INSERT INTO employee VALUES
    ('E1','Raul','Chong',95000,'D11','123-45-6789'),
    ('E2','Rav','Ahuja',75000,'D11','234-56-7890'),
    ('E3','Lakshmi','Katta',85000,'D20','345-67-8901')
""")
q("SELECT * FROM employee")

## 🧪 CREATE VIEW — a saved SELECT (no salary, no SSN)

In [ ]:
q("CREATE VIEW empinfo AS SELECT emp_id, f_name, l_name, dep_id FROM employee")

## 🧪 The view queries like a table

In [ ]:
q("SELECT * FROM empinfo")
q("SELECT f_name, dep_id FROM empinfo WHERE dep_id = 'D20'")

## 🧪 Views re-compute — live through the base table

In [ ]:
q("INSERT INTO employee VALUES ('E4','Ana','Silva',68000,'D30','456-78-9012')")
q("SELECT * FROM empinfo")   # Ana appears, although she wasn't there at CREATE time


## 🧪 Replace a view

SQLite has no `CREATE OR REPLACE VIEW` — the portable recipe is drop + create. On
DB2 / MySQL / PostgreSQL the same intent is one call: `CREATE OR REPLACE VIEW ...`.

In [ ]:
q("DROP VIEW empinfo")
q("CREATE VIEW empinfo AS SELECT emp_id, f_name, l_name, dep_id FROM employee WHERE dep_id <> 'D30'")
q("SELECT * FROM empinfo")

## 🧪 DROP the view

In [ ]:
q("DROP VIEW empinfo")
q("SELECT name, type FROM sqlite_master WHERE type = 'view'")

## 📕 Dialect notes

- View replacement: SQLite = DROP + CREATE; `CREATE OR REPLACE VIEW` works on
  DB2, MySQL, PostgreSQL, Oracle.
- **Rule from the course**: views shouldn't hide `ORDER BY` — ordering belongs to the
  outer query; some engines ignore it inside a view anyway.

## 💡 Tips & tricks

- Views hide sensitive columns (a free security layer) and save complex joins for reuse.
- Read-only here: SQLite only supports **simple** updatable views (INSERT THROUGH
  view is limited) — treat views as read models.

## 🎯 Your turn

1. Create `emp_salaries` view with names + salaries (this one is internal-only).
2. Query it for department D11, then replace the view to exclude D11 too.

In [ ]:
# 💡 Solution
q("CREATE VIEW emp_salaries AS SELECT f_name, l_name, salary, dep_id FROM employee")
q("SELECT * FROM emp_salaries WHERE dep_id = 'D11'")
q("DROP VIEW emp_salaries")
# SQLite has no CREATE OR REPLACE VIEW - the portable recipe is drop + create:
q("CREATE VIEW emp_salaries AS SELECT f_name, l_name, salary, dep_id FROM employee WHERE dep_id <> 'D11'")
q("SELECT * FROM emp_salaries")

# In engines that support it (DB2 / MySQL / PostgreSQL) the same intent is one line:
#   CREATE OR REPLACE VIEW emp_salaries AS SELECT ...